# 02 - AI triage (Phase 4 in Fabric)
Reads the `enquiries` table, triages each **masked** message, and writes `ai_triage` and `audit_log` tables.

**API key:** leave it out and this runs in **demo mode** (free, keyword stand-in). To use Claude, store the key in Azure Key Vault
and read it with `notebookutils.credentials.getSecret(...)`. Never paste a key into a cell. Remember Key Vault and API usage can cost money: delete them when you are finished.

In [ ]:
import sys
ROOT = "/lakehouse/default/Files"          # the attached Lakehouse's Files area
sys.path.insert(0, f"{ROOT}/src")          # so we can import the same .py files as the local edition
%pip install -q anthropic python-dotenv pydantic faker

In [ ]:
import os
# OPTIONAL real-Claude mode (uncomment, fill in YOUR vault URL and secret name):
# os.environ["ANTHROPIC_API_KEY"] = notebookutils.credentials.getSecret("https://YOUR-VAULT.vault.azure.net/", "anthropic-api-key")

import ai_client, triage
print("DEMO MODE" if ai_client.is_demo() else f"Real Claude: {ai_client.MODEL}")

In [ ]:
import pandas as pd
from datetime import datetime

enq = spark.table("enquiries").toPandas()
tracker = ai_client.CostTracker()
triage_rows, audit_rows = [], []
now = lambda: datetime.now().replace(microsecond=0).isoformat(sep=" ")

for _, r in enq.iterrows():
    masked = r["message_masked"] if isinstance(r["message_masked"], str) else ""
    result, error = None, "empty message"
    if masked.strip():
        if ai_client.is_demo():
            result, error = triage.demo_triage(masked), ""
        else:
            result, error = triage.claude_triage(masked, tracker)
    if result is None:
        triage_rows.append(dict(raw_row=int(r["raw_row"]), needs_human_review=1, reason=f"AI triage failed: {error}"))
        audit_rows.append(dict(ts=now(), raw_row=int(r["raw_row"]), actor="ai", action="triage_failed", detail=error))
        continue
    needs, why = triage.apply_review_rules(result)
    triage_rows.append(dict(raw_row=int(r["raw_row"]), category=result["category"], urgency=result["urgency"],
        summary=result["summary"], postcode=r["postcode"], requested_action=result["requested_action"],
        confidence=float(result["confidence"]), needs_human_review=int(needs), reason=result["reason"],
        mode="demo" if ai_client.is_demo() else "claude"))
    audit_rows.append(dict(ts=now(), raw_row=int(r["raw_row"]), actor="ai", action="triaged",
        detail=f"category={result['category']}; urgency={result['urgency']}; confidence={result['confidence']}"))
    if needs:
        audit_rows.append(dict(ts=now(), raw_row=int(r["raw_row"]), actor="system", action="queued_for_review", detail=why))

spark.createDataFrame(pd.DataFrame(triage_rows)).write.mode("overwrite").format("delta").saveAsTable("ai_triage")
spark.createDataFrame(pd.DataFrame(audit_rows)).write.mode("overwrite").format("delta").saveAsTable("audit_log")
print(len(triage_rows), "triaged;", sum(t["needs_human_review"] for t in triage_rows), "sent to review")
print(f"Claude cost so far: about GBP {tracker.cost_gbp:.4f}")